In [1]:
import pandas as pd
import numpy as np
import pickle
import optuna
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, precision_recall_curve, f1_score
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from imblearn.combine import SMOTEENN

optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\supra\OneDrive\Documents\Road_Closure\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sklearn
print(sklearn.__version__)

1.7.2


In [3]:
df=pd.read_csv("data.csv")

In [4]:
# ── Constants ─────────────────────────────────────────────────────────────────
MODEL_PATH     = "road_closure_model.pkl"
PEAK_HOURS     = {7, 8, 9, 17, 18, 19, 20, 21}
NIGHT_HOURS    = set(range(22, 24)) | set(range(0, 5))
N_GEO_CLUSTERS = 12
CAT_COLS       = ['event_cause', 'priority', 'veh_type', 'corridor', 'zone', 'event_type']

FEATURES = [
    # Categorical
    'event_cause', 'priority', 'veh_type', 'corridor', 'zone', 'event_type',
    # Raw time
    'hour', 'day_of_week',
    # Cyclical encoding
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    # Binary flags
    'is_peak_hour', 'is_weekend', 'is_night',
    # Geographic cluster
    'geo_cluster',
    # Corridor traffic density
    'corridor_volume',
    # Historical closure rates (target encoding)
    'cause_closure_rate', 'corridor_closure_rate',
    # Interaction features
    'cause_x_peak', 'priority_x_unplanned', 'cause_x_corridor_vol', 'priority_num',
    # ── Domain features (new) ─────────────────────────────────────────────
    'is_heavy_vehicle', 'has_cargo', 'duration_hours', 'is_old_truck', 'at_junction',
    'has_police', 'is_accident', 'has_direction', 'is_both_directions',
    'is_serious_breakdown', 'is_high_risk_zone', 'time_to_resolve_hours',
    'resolved_at_diff_location', 'has_comment', 'has_metadata',
]

In [5]:
# ── Feature Engineering Helpers ───────────────────────────────────────────────
def clean_categoricals(df):
    df['veh_type']    = df['veh_type'].replace('', 'unknown').fillna('unknown')
    df['zone']        = df['zone'].replace('NULL', 'unknown').fillna('unknown')
    df['corridor']    = df['corridor'].fillna('Non-corridor')
    df['event_cause'] = df['event_cause'].fillna('others')
    df['priority']    = df['priority'].replace('NULL', 'Low').fillna('Low')
    df['event_type']  = df['event_type'].fillna('unplanned')
    return df


def add_time_features(df):
    df['start_datetime'] = pd.to_datetime(df['start_datetime'], utc=True, errors='coerce')
    df['hour']        = df['start_datetime'].dt.hour.fillna(0).astype(int)
    df['day_of_week'] = df['start_datetime'].dt.dayofweek.fillna(0).astype(int)
    df['month']       = df['start_datetime'].dt.month.fillna(1).astype(int)
    df['hour_sin']    = np.sin(2 * np.pi * df['hour']        / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour']        / 24)
    df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['month']       / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month']       / 12)
    df['is_peak_hour'] = df['hour'].isin(PEAK_HOURS).astype(int)
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_night']     = df['hour'].isin(NIGHT_HOURS).astype(int)
    return df


def add_geo_features(df, kmeans=None, fit=False):
    coords = df[['latitude', 'longitude']].fillna(0)
    if fit:
        kmeans = KMeans(n_clusters=N_GEO_CLUSTERS, random_state=42, n_init=10)
        df['geo_cluster'] = kmeans.fit_predict(coords)
        return df, kmeans
    df['geo_cluster'] = kmeans.predict(coords)
    return df


def add_statistical_features(df, stats=None, fit=False):
    if fit:
        stats = {
            'cause_rate':    df.groupby('event_cause')['road_closure'].mean().to_dict(),
            'corridor_rate': df.groupby('corridor')['road_closure'].mean().to_dict(),
            'corridor_vol':  df['corridor'].value_counts().to_dict(),
        }
    df['cause_closure_rate']    = df['event_cause'].map(stats['cause_rate']).fillna(0)
    df['corridor_closure_rate'] = df['corridor'].map(stats['corridor_rate']).fillna(0)
    df['corridor_volume']       = df['corridor'].map(stats['corridor_vol']).fillna(0)
    return (df, stats) if fit else df


def add_interaction_features(df):
    df['priority_num']         = df['priority'].map({'High': 1, 'Low': 0}).fillna(0)
    df['cause_x_peak']         = df['cause_closure_rate'] * df['is_peak_hour']
    df['priority_x_unplanned'] = df['priority_num'] * (df['event_type'] == 'unplanned').astype(int)
    df['cause_x_corridor_vol'] = df['cause_closure_rate'] * np.log1p(df['corridor_volume'])
    return df


def add_domain_features(df):
    """Causally-motivated features using previously unused columns."""

    # Vehicle type
    df['is_heavy_vehicle'] = (
        df['veh_type'].astype(str).str.lower()
        .str.contains('truck|lorry|bus|tanker|trailer|heavy', na=False)
        .astype(int)
    )

    # Cargo presence
    df['has_cargo'] = df['cargo_material'].notna().astype(int) \
        if 'cargo_material' in df.columns else 0

    # Incident duration
    start = pd.to_datetime(df['start_datetime'], utc=True, errors='coerce')
    end   = pd.to_datetime(df.get('end_datetime', pd.NaT), utc=True, errors='coerce')
    df['duration_hours'] = (
        (end - start).dt.total_seconds() / 3600
    ).clip(0, 48).fillna(0)

    # Old truck
    df['age_of_truck'] = pd.to_numeric(
        df.get('age_of_truck', 0), errors='coerce'
    ).fillna(0)
    df['is_old_truck'] = (df['age_of_truck'] > 10).astype(int)

    # Junction
    df['at_junction'] = df['junction'].notna().astype(int) \
        if 'junction' in df.columns else 0

    # Police involvement
    df['has_police'] = df['assigned_to_police_id'].notna().astype(int) \
        if 'assigned_to_police_id' in df.columns else 0

    # Citizen-reported accident
    df['is_accident'] = df['citizen_accident_id'].notna().astype(int) \
        if 'citizen_accident_id' in df.columns else 0

    # Direction signals
    direction = df['direction'].astype(str).str.lower() \
        if 'direction' in df.columns else pd.Series([''] * len(df))
    df['has_direction']      = (direction != 'nan').astype(int)
    df['is_both_directions'] = direction.str.contains(
        'both|all|contra', na=False
    ).astype(int)

    # Serious breakdown reason
    df['is_serious_breakdown'] = (
        df.get('reason_breakdown', pd.Series([''] * len(df)))
        .astype(str).str.lower()
        .str.contains('fire|accident|collision|engine|oil|flood', na=False)
        .astype(int)
    )

    # High-risk zone
    df['is_high_risk_zone'] = (
        df['zone'].astype(str).str.lower()
        .str.contains('highway|tunnel|bridge|school|hospital', na=False)
        .astype(int)
    )

    # Time to resolve
    resolved = pd.to_datetime(
        df.get('resolved_datetime', pd.NaT), utc=True, errors='coerce'
    )
    df['time_to_resolve_hours'] = (
        (resolved - start).dt.total_seconds() / 3600
    ).clip(0, 72).fillna(0)

    # Resolution at different location
    if 'resolved_at_address' in df.columns and 'address' in df.columns:
        df['resolved_at_diff_location'] = (
            df['resolved_at_address'].notna() &
            (df['resolved_at_address'] != df['address'])
        ).astype(int)
    else:
        df['resolved_at_diff_location'] = 0

    # Comment / metadata presence
    df['has_comment']  = df['comment'].notna().astype(int)  \
        if 'comment'   in df.columns else 0
    df['has_metadata'] = df['meta_data'].notna().astype(int) \
        if 'meta_data' in df.columns else 0

    return df

In [6]:
# ── Full Preprocessing Pipeline ───────────────────────────────────────────────
def preprocess(df, encoders=None, imputer=None, kmeans=None, stats=None, fit=False):
    df = df.copy()
    df = clean_categoricals(df)
    df = add_time_features(df)

    if fit:
        df, kmeans = add_geo_features(df, fit=True)
        df, stats  = add_statistical_features(df, fit=True)
    else:
        df = add_geo_features(df, kmeans=kmeans)
        df = add_statistical_features(df, stats=stats)

    df = add_interaction_features(df)
    df = add_domain_features(df)

    X = df[FEATURES].copy()

    if fit:
        encoders = {}
        for col in CAT_COLS:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            encoders[col] = le
        imputer = SimpleImputer(strategy='median')
        X = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)
        return X, encoders, imputer, kmeans, stats
    else:
        for col in CAT_COLS:
            le = encoders[col]
            X[col] = X[col].astype(str).apply(
                lambda v: le.transform([v])[0] if v in le.classes_ else -1
            )
        X = pd.DataFrame(imputer.transform(X), columns=FEATURES)
        return X

In [7]:
# ── Hyperparameter Tuning Helpers ─────────────────────────────────────────────
def _cv_f1(model, X, y, n_splits=5):
    """Stratified CV with per-fold SMOTEENN. Returns mean Road Closure F1."""
    cv, scores = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42), []
    for tr, va in cv.split(X, y):
        Xf, Xv = X.iloc[tr], X.iloc[va]
        yf, yv = y.iloc[tr], y.iloc[va]
        Xf_r, yf_r = SMOTEENN(random_state=42).fit_resample(Xf, yf)
        model.fit(Xf_r, yf_r)
        yv_p = model.predict_proba(Xv)[:, 1]
        p, r, t = precision_recall_curve(yv, yv_p)
        f1s  = 2 * p * r / (p + r + 1e-9)
        best = t[np.argmax(f1s[:-1])] if len(t) else 0.5
        scores.append(f1_score(yv, (yv_p >= best).astype(int)))
    return np.mean(scores)


def _tune_xgb(X_train, y_train, spw, n_trials=25):
    def obj(trial):
        return _cv_f1(XGBClassifier(
            n_estimators     = trial.suggest_int  ("n_estimators",     100, 500, step=50),
            max_depth        = trial.suggest_int  ("max_depth",        3, 8),
            learning_rate    = trial.suggest_float("learning_rate",    0.01, 0.2,  log=True),
            subsample        = trial.suggest_float("subsample",        0.5, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight = trial.suggest_int  ("min_child_weight", 1, 20),
            gamma            = trial.suggest_float("gamma",            0.0, 5.0),
            reg_alpha        = trial.suggest_float("reg_alpha",        1e-4, 10.0, log=True),
            reg_lambda       = trial.suggest_float("reg_lambda",       1e-4, 10.0, log=True),
            scale_pos_weight = trial.suggest_int  ("scale_pos_weight", spw // 2, spw * 2),
            random_state=42, eval_metric="aucpr", verbosity=0,
        ), X_train, y_train)
    s = optuna.create_study(direction="maximize")
    s.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  XGBoost  best CV F1: {s.best_value:.4f}")
    return s.best_params


def _tune_lgbm(X_train, y_train, n_trials=25):
    def obj(trial):
        return _cv_f1(LGBMClassifier(
            n_estimators      = trial.suggest_int  ("n_estimators",     100, 500, step=50),
            max_depth         = trial.suggest_int  ("max_depth",        3, 8),
            learning_rate     = trial.suggest_float("learning_rate",    0.01, 0.2,  log=True),
            subsample         = trial.suggest_float("subsample",        0.5, 1.0),
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_samples = trial.suggest_int  ("min_child_samples",5, 50),
            reg_alpha         = trial.suggest_float("reg_alpha",        1e-4, 10.0, log=True),
            reg_lambda        = trial.suggest_float("reg_lambda",       1e-4, 10.0, log=True),
            num_leaves        = trial.suggest_int  ("num_leaves",       20, 100),
            class_weight="balanced", random_state=42, verbosity=-1,
        ), X_train, y_train)
    s = optuna.create_study(direction="maximize")
    s.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  LightGBM best CV F1: {s.best_value:.4f}")
    return s.best_params


def _tune_rf(X_train, y_train, n_trials=20):
    def obj(trial):
        return _cv_f1(RandomForestClassifier(
            n_estimators      = trial.suggest_int  ("n_estimators",     100, 500, step=50),
            max_depth         = trial.suggest_int  ("max_depth",        4, 12),
            min_samples_split = trial.suggest_int  ("min_samples_split",2, 20),
            min_samples_leaf  = trial.suggest_int  ("min_samples_leaf", 1, 20),
            max_features      = trial.suggest_float("max_features",     0.3, 1.0),
            class_weight="balanced_subsample", random_state=42, n_jobs=-1,
        ), X_train, y_train)
    s = optuna.create_study(direction="maximize")
    s.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  RF       best CV F1: {s.best_value:.4f}")
    return s.best_params


def _tune_weights(X_train, y_train, xgb_p, lgbm_p, rf_p, n_trials=15):
    def obj(trial):
        w = [trial.suggest_float(f"w_{m}", 0.5, 5.0) for m in ('xgb','lgbm','rf')]
        ens = VotingClassifier(estimators=[
            ('xgb',  XGBClassifier(**xgb_p,  random_state=42, eval_metric="aucpr", verbosity=0)),
            ('lgbm', LGBMClassifier(**lgbm_p, random_state=42, class_weight="balanced", verbosity=-1)),
            ('rf',   RandomForestClassifier(**rf_p, random_state=42,
                                             class_weight="balanced_subsample", n_jobs=-1)),
        ], voting='soft', weights=w)
        return _cv_f1(ens, X_train, y_train)
    s = optuna.create_study(direction="maximize")
    s.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    print(f"  Ensemble best CV F1: {s.best_value:.4f}")
    return [s.best_params['w_xgb'], s.best_params['w_lgbm'], s.best_params['w_rf']]


In [8]:
# ── Training ──────────────────────────────────────────────────────────────────
def train(csv_path: str):
    print("Loading data...")
    df = pd.read_csv(csv_path)
    df['road_closure'] = df['requires_road_closure'].astype(int)

    X, encoders, imputer, kmeans, stats = preprocess(df, fit=True)
    y = df['road_closure']
    print(f"Samples: {len(y)} | Closure rate: {y.mean()*100:.1f}% | Features: {len(FEATURES)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    spw      = int(neg / pos)

    # ── Phase 1-4: Tune each model then ensemble weights ──────────────────
    print("\n── Tuning XGBoost ──────────────────────────────────────────────")
    xgb_params = _tune_xgb(X_train, y_train, spw, n_trials=25)

    print("\n── Tuning LightGBM ─────────────────────────────────────────────")
    lgbm_params = _tune_lgbm(X_train, y_train, n_trials=25)

    print("\n── Tuning RandomForest ─────────────────────────────────────────")
    rf_params = _tune_rf(X_train, y_train, n_trials=20)

    print("\n── Tuning Ensemble Weights ─────────────────────────────────────")
    best_weights = _tune_weights(X_train, y_train, xgb_params, lgbm_params, rf_params, n_trials=15)
    print(f"  Weights → XGB={best_weights[0]:.2f} | LGBM={best_weights[1]:.2f} | RF={best_weights[2]:.2f}")

    # ── Phase 5: Retrain on full training set ─────────────────────────────
    print("\n── Retraining final ensemble ───────────────────────────────────")
    X_res, y_res = SMOTEENN(random_state=42).fit_resample(X_train, y_train)
    print(f"  Resampled → {dict(zip(*np.unique(y_res, return_counts=True)))}")

    model = VotingClassifier(estimators=[
        ('xgb',  XGBClassifier(**xgb_params,  random_state=42, eval_metric="aucpr", verbosity=0)),
        ('lgbm', LGBMClassifier(**lgbm_params, random_state=42, class_weight="balanced", verbosity=-1)),
        ('rf',   RandomForestClassifier(**rf_params, random_state=42,
                                         class_weight="balanced_subsample", n_jobs=-1)),
    ], voting='soft', weights=best_weights)
    model.fit(X_res, y_res)

    # ── Phase 6: Optimal threshold ────────────────────────────────────────
    y_proba = model.predict_proba(X_test)[:, 1]
    prec, rec, thresholds = precision_recall_curve(y_test, y_proba)

    valid = np.where((rec[:-1] >= 0.65) & (prec[:-1] >= 0.35))[0]
    if len(valid):
        f1s         = 2 * prec * rec / (prec + rec + 1e-9)
        best_thresh = thresholds[valid[np.argmax(f1s[valid])]]
    else:
        f1s         = 2 * prec * rec / (prec + rec + 1e-9)
        best_thresh = thresholds[np.argmax(f1s[:-1])]

    print(f"\n  Optimal threshold: {best_thresh:.3f}")

    # ── Results ───────────────────────────────────────────────────────────
    y_pred = (y_proba >= best_thresh).astype(int)
    print("\n=== Results ===")
    print(classification_report(y_test, y_pred,
                                 target_names=['No Closure', 'Road Closure']))

    print("Top Feature Importances (XGBoost member):")
    xgb_fitted = model.named_estimators_['xgb']
    for f, i in sorted(zip(FEATURES, xgb_fitted.feature_importances_),
                        key=lambda x: -x[1])[:10]:
        print(f"  {f}: {i:.3f}")

    # ── Save ──────────────────────────────────────────────────────────────
    '''bundle = dict(
        model       = model,
        encoders    = encoders,
        imputer     = imputer,
        kmeans      = kmeans,
        stats       = stats,
        features    = FEATURES,
        threshold   = best_thresh,
        best_params = {'xgb': xgb_params, 'lgbm': lgbm_params,
                       'rf': rf_params, 'weights': best_weights}
    )
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(bundle, f)
    print(f"\nSaved → {MODEL_PATH}")
    return bundle'''

In [9]:

def load_model():
    with open(MODEL_PATH, 'rb') as f:
        return pickle.load(f)

In [10]:
def predict_road_closure(
    event_cause     : str,
    priority        : str,
    veh_type        : str,
    corridor        : str,
    zone            : str,
    latitude        : float,
    longitude       : float,
    hour            : int,
    day_of_week     : int,
    event_type      : str,
    # optional extra fields used by domain features
    cargo_material  : str   = None,
    age_of_truck    : float = None,
    junction        : str   = None,
    assigned_to_police_id : str = None,
    citizen_accident_id   : str = None,
    direction       : str   = None,
    reason_breakdown: str   = None,
    comment         : str   = None,
    meta_data       : str   = None,
    end_datetime    : str   = None,
    resolved_datetime: str  = None,
    resolved_at_address: str = None,
    address         : str   = None,
    bundle          : dict  = None,
) -> dict:
    """
    Predict whether a traffic incident needs road closure.

    Returns:
        road_closure_required : bool
        confidence            : float 0-1
        risk_level            : 'Low' | 'Medium' | 'High'
    """
    if bundle is None:
        bundle = load_model()

    now = pd.Timestamp.now(tz='UTC').replace(hour=hour, minute=0, second=0)

    row = pd.DataFrame([{
        'start_datetime'       : now,
        'end_datetime'         : end_datetime,
        'resolved_datetime'    : resolved_datetime,
        'event_cause'          : event_cause,
        'priority'             : priority,
        'veh_type'             : veh_type,
        'corridor'             : corridor,
        'zone'                 : zone,
        'event_type'           : event_type,
        'latitude'             : latitude,
        'longitude'            : longitude,
        'cargo_material'       : cargo_material,
        'age_of_truck'         : age_of_truck,
        'junction'             : junction,
        'assigned_to_police_id': assigned_to_police_id,
        'citizen_accident_id'  : citizen_accident_id,
        'direction'            : direction,
        'reason_breakdown'     : reason_breakdown,
        'comment'              : comment,
        'meta_data'            : meta_data,
        'resolved_at_address'  : resolved_at_address,
        'address'              : address,
        'road_closure'         : 0,
    }])

    X    = preprocess(row, encoders=bundle['encoders'], imputer=bundle['imputer'],
                      kmeans=bundle['kmeans'], stats=bundle['stats'])
    prob = float(bundle['model'].predict_proba(X)[0][1])
    thresh = bundle.get('threshold', 0.5)

    return {
        "road_closure_required": prob >= thresh,
        "confidence"           : round(prob, 3),
        "risk_level"           : "Low" if prob < 0.3 else "Medium" if prob < 0.6 else "High"
    }

In [11]:
# ── Run directly ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    csv    = "data.csv"
    bundle = train(csv)

    print("\n=== Sample Predictions ===")
    samples = [
        dict(event_cause="tree_fall",         priority="High", veh_type="unknown",
             corridor="Mysore Road",    zone="Central Zone 2", latitude=12.97, longitude=77.59,
             hour=8,  day_of_week=1, event_type="unplanned", reason_breakdown="fallen_tree",
             assigned_to_police_id="P001"),

        dict(event_cause="vehicle_breakdown", priority="Low",  veh_type="private_car",
             corridor="Non-corridor",   zone="unknown",        latitude=13.02, longitude=77.55,
             hour=14, day_of_week=3, event_type="unplanned"),

        dict(event_cause="vip_movement",      priority="High", veh_type="unknown",
             corridor="Bellary Road 1", zone="North Zone 1",   latitude=13.07, longitude=77.58,
             hour=18, day_of_week=0, event_type="planned", direction="both"),

        dict(event_cause="construction",      priority="High", veh_type="heavy_vehicle",
             corridor="Tumkur Road",    zone="West Zone 1",    latitude=13.04, longitude=77.51,
             hour=9,  day_of_week=2, event_type="planned", cargo_material="cement",
             age_of_truck=12, reason_breakdown="engine_failure"),
    ]

    for s in samples:
        r = predict_road_closure(**s, bundle=bundle)
        print(f"  {s['event_cause']:20s} | {s['priority']:4s} | h={s['hour']:2d}"
              f" → Closure: {str(r['road_closure_required']):5s}"
              f" | Conf: {r['confidence']:.2f} | Risk: {r['risk_level']}")

Loading data...
Samples: 8173 | Closure rate: 8.3% | Features: 40

── Tuning XGBoost ──────────────────────────────────────────────


Best trial: 17. Best value: 0.457347: 100%|██████████| 25/25 [00:52<00:00,  2.10s/it]


  XGBoost  best CV F1: 0.4573

── Tuning LightGBM ─────────────────────────────────────────────


Best trial: 16. Best value: 0.460359: 100%|██████████| 25/25 [00:40<00:00,  1.61s/it]


  LightGBM best CV F1: 0.4604

── Tuning RandomForest ─────────────────────────────────────────


Best trial: 13. Best value: 0.44981: 100%|██████████| 20/20 [02:54<00:00,  8.72s/it] 


  RF       best CV F1: 0.4498

── Tuning Ensemble Weights ─────────────────────────────────────


Best trial: 11. Best value: 0.463837: 100%|██████████| 15/15 [03:45<00:00, 15.01s/it]


  Ensemble best CV F1: 0.4638
  Weights → XGB=0.75 | LGBM=3.86 | RF=0.54

── Retraining final ensemble ───────────────────────────────────
  Resampled → {np.int64(0): np.int64(4531), np.int64(1): np.int64(5484)}

  Optimal threshold: 0.526

=== Results ===
              precision    recall  f1-score   support

  No Closure       0.95      0.96      0.95      1500
Road Closure       0.49      0.41      0.44       135

    accuracy                           0.92      1635
   macro avg       0.72      0.68      0.70      1635
weighted avg       0.91      0.92      0.91      1635

Top Feature Importances (XGBoost member):
  has_police: 0.104
  cause_closure_rate: 0.087
  cause_x_corridor_vol: 0.082
  is_weekend: 0.060
  dow_sin: 0.045
  is_heavy_vehicle: 0.044
  day_of_week: 0.040
  is_night: 0.036
  dow_cos: 0.036
  at_junction: 0.034

=== Sample Predictions ===
  tree_fall            | High | h= 8 → Closure: True  | Conf: 0.96 | Risk: High
  vehicle_breakdown    | Low  | h=14 → Closure: 